# 从零实现 N-BEATS：Backcast、Forecast 与可解释 Basis

N-BEATS 用一串 block 反复解释输入历史：每个 block 输出 backcast，从 residual 中扣除；同时输出 forecast 并累加。本 Notebook 手写 generic/trend/seasonality basis、残差堆叠、严格时间窗口、直接多步预测、评估与发布 wrapper。

合成趋势+双季节序列只验证结构能学习已知模式，不代表真实需求预测。生产系统还要处理缺失、节假日、协变量、层级一致性、概念漂移和概率区间。

In [ ]:
import copy,hashlib,io,json,math,random,warnings
from types import MappingProxyType
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
SEED65=6501
random.seed(SEED65); np.random.seed(SEED65); torch.manual_seed(SEED65); torch.set_num_threads(1)
def canonical65(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"))
def sha65(x): return hashlib.sha256(x).hexdigest()
assert torch.get_num_threads()==1

## 1. 严格时间切分与窗口

序列 $y_t=0.004t+\sin(2\pi t/24)+0.3\sin(2\pi t/7)+\epsilon_t$。前 360 点 train，接着 96 点 validation，最后 96 点 test。normalizer 只用 train。

backcast 长 36、forecast 长 6。一个窗口允许读取 forecast 起点以前的历史，但所有 target 必须完整落在所属 split；禁止随机打散后切分造成未来泄漏。

In [ ]:
def make_series65(length=552,seed=SEED65+1):
    g=torch.Generator().manual_seed(seed); t=torch.arange(length,dtype=torch.float32)
    return .004*t+torch.sin(2*math.pi*t/24)+.3*torch.sin(2*math.pi*t/7)+.04*torch.randn(length,generator=g)
raw65=make_series65(); train_end65=360; val_end65=456; mean65=raw65[:train_end65].mean(); std65=raw65[:train_end65].std(unbiased=False)
series65=(raw65-mean65)/std65; BACK65=36; HORIZON65=6
def windows65(start,end):
    xs=[]; ys=[]; origins=[]
    for origin in range(max(start,BACK65),end-HORIZON65+1):
        xs.append(series65[origin-BACK65:origin]); ys.append(series65[origin:origin+HORIZON65]); origins.append(origin)
    return torch.stack(xs),torch.stack(ys),torch.tensor(origins)
train_x65,train_y65,train_o65=windows65(BACK65,train_end65); val_x65,val_y65,val_o65=windows65(train_end65,val_end65); test_x65,test_y65,test_o65=windows65(val_end65,len(raw65))
assert train_x65.shape[1:]==(BACK65,) and train_y65.shape[1:]==(HORIZON65,)
assert train_o65.max()+HORIZON65<=train_end65 and val_o65.min()>=train_end65 and test_o65.min()>=val_end65
assert torch.allclose(raw65,make_series65()) and std65>0

## 2. Generic、Trend 与 Seasonality basis

block 的 MLP 输出系数 $\theta$，basis 把系数变为 backcast/forecast。Generic 直接把前后系数视为序列；trend 使用归一化时间的多项式；seasonality 使用 Fourier sin/cos。

可解释性来自限制 basis，而不是给任意 MLP 输出贴“趋势”标签。shape 合同分别是 `[B,theta_dim] -> ([B,36],[B,6])`。

In [ ]:
class GenericBasis65(nn.Module):
    def __init__(self,backcast,forecast): super().__init__(); self.backcast=backcast; self.forecast=forecast; self.theta_dim=backcast+forecast
    def forward(self,theta):
        if theta.ndim!=2 or theta.shape[1]!=self.theta_dim: raise ValueError("generic_theta_contract")
        return theta[:,:self.backcast],theta[:,self.backcast:]
class TrendBasis65(nn.Module):
    def __init__(self,backcast,forecast,degree=2):
        super().__init__(); self.order=degree+1; self.theta_dim=2*self.order
        tb=torch.linspace(-1,0,backcast); tf=torch.linspace(0,1,forecast)
        self.register_buffer("back_basis",torch.stack([tb**i for i in range(self.order)])); self.register_buffer("fore_basis",torch.stack([tf**i for i in range(self.order)]))
    def forward(self,theta):
        if theta.ndim!=2 or theta.shape[1]!=self.theta_dim: raise ValueError("trend_theta_contract")
        b,f=theta.chunk(2,-1); return b@self.back_basis,f@self.fore_basis
class SeasonalityBasis65(nn.Module):
    def __init__(self,backcast,forecast,harmonics=3,period=24.):
        super().__init__()
        if harmonics<1 or not math.isfinite(period) or period<=0: raise ValueError("seasonality_config_contract")
        self.harmonics=harmonics; self.period=float(period); self.theta_dim=4*harmonics
        def basis(times):
            freq=torch.arange(1,harmonics+1,dtype=torch.float32)[:,None]
            phase=2*math.pi*freq*times[None,:]/self.period
            return torch.cat([torch.cos(phase),torch.sin(phase)],0)
        self.register_buffer("back_basis",basis(torch.arange(-backcast,0,dtype=torch.float32))); self.register_buffer("fore_basis",basis(torch.arange(forecast,dtype=torch.float32)))
    def forward(self,theta):
        if theta.ndim!=2 or theta.shape[1]!=self.theta_dim: raise ValueError("season_theta_contract")
        b,f=theta.chunk(2,-1); return b@self.back_basis,f@self.fore_basis
trend_probe65=TrendBasis65(3,3,1); theta_probe65=torch.tensor([[2.,1.,3.,-1.]])
back_probe65,fore_probe65=trend_probe65(theta_probe65)
assert torch.allclose(back_probe65,torch.tensor([[1.,1.5,2.]])) and torch.allclose(fore_probe65,torch.tensor([[3.,2.5,2.]]))
generic_probe65=GenericBasis65(2,1); assert generic_probe65(torch.tensor([[1.,2.,3.]]))[1].item()==3
season_probe65=SeasonalityBasis65(3,2,1,period=4.); season_theta65=torch.tensor([[1.,0.,1.,0.]])
season_back65,season_fore65=season_probe65(season_theta65)
assert torch.allclose(season_back65,torch.cos(2*math.pi*torch.arange(-3,0)/4.)[None,:],atol=1e-6)
assert torch.allclose(season_fore65,torch.cos(2*math.pi*torch.arange(2)/4.)[None,:],atol=1e-6)

## 3. Block 与双残差堆叠

每个 block 用四层 MLP 将 residual `[B,36]` 映射到 theta，再由 basis 输出 `(backcast,forecast)`。模型递推：
$$r_{l+1}=r_l-\hat x_l,\qquad \hat y=\sum_l\hat y_l.$$

这称为 doubly residual stacking：输入残差向前传，forecast 残差横向累加。返回各 block component 便于诊断。

In [ ]:
class NBeatsBlock65(nn.Module):
    def __init__(self,input_size,basis,hidden=64):
        super().__init__(); self.input_size=input_size; self.basis=basis
        self.mlp=nn.Sequential(nn.Linear(input_size,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,basis.theta_dim))
    def forward(self,x):
        if x.ndim!=2 or x.shape[1]!=self.input_size or not torch.isfinite(x).all(): raise ValueError("block_input_contract")
        return self.basis(self.mlp(x))
class NBeats65(nn.Module):
    def __init__(self,blocks,forecast_size): super().__init__(); self.blocks=nn.ModuleList(blocks); self.forecast_size=forecast_size
    def forward(self,x,return_components=False):
        residual=x; forecast=torch.zeros(x.shape[0],self.forecast_size,dtype=x.dtype,device=x.device); components=[]
        for block in self.blocks:
            back,fore=block(residual); residual=residual-back; forecast=forecast+fore; components.append(fore)
        if not torch.isfinite(residual).all() or not torch.isfinite(forecast).all(): raise ValueError("nonfinite_nbeats_output")
        return (forecast,residual,components) if return_components else forecast
zero_basis65=GenericBasis65(BACK65,HORIZON65); zero_block65=NBeatsBlock65(BACK65,zero_basis65,8)
with torch.no_grad():
    for p in zero_block65.parameters(): p.zero_()
zero_model65=NBeats65([zero_block65],HORIZON65); zf65,zr65,zc65=zero_model65(train_x65[:2],True)
assert torch.equal(zf65,torch.zeros_like(zf65)) and torch.equal(zr65,train_x65[:2]) and len(zc65)==1

## 4. 可解释 stack 的受控训练

使用 trend、seasonality、generic 三个 block。训练只最小化 train 窗口 MSE；每 40 步查看 validation，但 test 最后一次报告。基线是把历史最后一个值复制 6 步。

真实 N-BEATS 常用更深 stack、不同 loss 和 ensemble。本例缩小网络以在 CPU 快速验证 residual 与 basis。

In [ ]:
torch.manual_seed(SEED65)
model65=NBeats65([NBeatsBlock65(BACK65,TrendBasis65(BACK65,HORIZON65,2)),NBeatsBlock65(BACK65,SeasonalityBasis65(BACK65,HORIZON65,4)),NBeatsBlock65(BACK65,GenericBasis65(BACK65,HORIZON65))],HORIZON65)
opt65=torch.optim.Adam(model65.parameters(),lr=3e-3); initial65=float(F.mse_loss(model65(val_x65),val_y65)); history65=[]
for step65 in range(440):
    pred65=model65(train_x65); loss65=F.mse_loss(pred65,train_y65); opt65.zero_grad(set_to_none=True); loss65.backward(); torch.nn.utils.clip_grad_norm_(model65.parameters(),5.); opt65.step()
    if step65%40==0: history65.append(float(F.mse_loss(model65(val_x65),val_y65)))
with torch.no_grad(): val_pred65=model65(val_x65); test_pred65=model65(test_x65)
val_mae65=float((val_pred65-val_y65).abs().mean()); test_mae65=float((test_pred65-test_y65).abs().mean()); naive_mae65=float((test_x65[:,-1,None]-test_y65).abs().mean())
assert val_mae65<.22 and test_mae65<naive_mae65*.65 and history65[-1]<initial65
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model65.parameters())
print({"val_mae":round(val_mae65,4),"test_mae":round(test_mae65,4),"naive":round(naive_mae65,4)})

## 5. 组件、尺度与未来干预 oracle

component 和应等于总 forecast；改变一个样本不会影响其他 batch 行。发布接口必须接收原始单位 36 点历史，在内部使用 train-only mean/std，再把 6 步输出反标准化。

时间模型的“未来隔离”首先由窗口 builder 保证；网络只看到固定 backcast，不存在把 target 拼入特征的通道。

In [ ]:
with torch.no_grad(): forecast65,residual65,components65=model65(test_x65[:4],True)
assert torch.allclose(torch.stack(components65).sum(0),forecast65,atol=1e-6) and residual65.shape==(4,BACK65)
changed65=test_x65[:4].clone(); changed65[0]+=1.; changed_pred65=model65(changed65)
assert torch.allclose(changed_pred65[1:],forecast65[1:],atol=1e-6) and not torch.allclose(changed_pred65[0],forecast65[0])
try: model65(torch.zeros(2,BACK65-1)); raise AssertionError("wrong history accepted")
except ValueError as e: assert str(e)=="block_input_contract"

### 5.1 滚动原点指标、逐 horizon 误差与代数 oracle

多步预测不能只汇报一个平均数：第 1 步很好、第 6 步崩溃时，整体 MAE 会掩盖问题。下面给出逐 horizon MAE、训练集一步朴素尺度上的 MASE，以及中位数 pinball loss。MASE 的分母只能从训练区间计算，否则仍然属于评估泄漏。

同时手工重放每个 block 的 `residual -= backcast` 与 `forecast += component`，验证框架前向确实实现了论文公式；再用 batch 置换证明样本之间没有隐式串扰。

In [ ]:
horizon_mae65=(test_pred65-test_y65).abs().mean(0)
naive_horizon_mae65=(test_x65[:,-1,None]-test_y65).abs().mean(0)
train_naive_scale65=(series65[1:train_end65]-series65[:train_end65-1]).abs().mean()
mase65=(test_pred65-test_y65).abs().mean()/train_naive_scale65
error65=test_y65-test_pred65; pinball50_65=torch.maximum(.5*error65,-.5*error65).mean()
assert horizon_mae65.shape==(HORIZON65,) and naive_horizon_mae65.shape==(HORIZON65,)
assert torch.isfinite(horizon_mae65).all() and bool((horizon_mae65>=0).all())
assert torch.isclose(horizon_mae65.mean(),torch.tensor(test_mae65),atol=1e-7)
assert train_naive_scale65>0 and torch.isfinite(mase65) and mase65>0
assert torch.allclose(pinball50_65,.5*(test_pred65-test_y65).abs().mean(),atol=1e-7)
manual_residual65=test_x65[:5].clone(); manual_forecast65=torch.zeros(5,HORIZON65)
for block65 in model65.blocks:
    manual_back65,manual_fore65=block65(manual_residual65); manual_residual65-=manual_back65; manual_forecast65+=manual_fore65
api_forecast65,api_residual65,_=model65(test_x65[:5],True)
assert torch.allclose(manual_forecast65,api_forecast65,atol=1e-6)
assert torch.allclose(manual_residual65,api_residual65,atol=1e-6)
perm65=torch.tensor([3,0,4,1,2])
assert torch.allclose(model65(test_x65[:5][perm65]),api_forecast65[perm65],atol=1e-6)

## 6. Published forecaster

manifest 绑定公式/seed、时间边界、window、normalizer、basis 顺序、训练 recipe 与测试协议。loader 重新生成序列并推导 train mean/std。`PublishedNBeats65.forecast` 只接受原始单位 `[B,36]`，返回原单位 `[B,6]`。

包外 registry 拒绝整体重签；state 摘要包含 key/dtype/shape/bytes。

In [ ]:
def th65(t):
    v=t.detach().cpu().contiguous(); return sha65(str(v.dtype).encode()+canonical65(list(v.shape)).encode()+v.numpy().tobytes())
def sd65(state):
    h=hashlib.sha256()
    for k,v in sorted(state.items()): h.update(k.encode()); h.update(th65(v).encode())
    return h.hexdigest()
config65={"input_size":BACK65,"forecast_size":HORIZON65,"blocks":[["trend",2],["seasonality",4],["generic",0]],"seasonality_base_period":24.,"hidden":64}
manifest65={"artifact_id":"nbeats-synthetic-v1","version":1,"model_config":config65,"data":{"length":552,"seed":SEED65+1,"train_end":train_end65,"val_end":val_end65,"snapshot":th65(raw65)},"preprocess":{"mean":float(mean65),"std":float(std65)},"training":{"seed":SEED65,"optimizer":"Adam","steps":440,"lr":.003,"grad_clip":5.,"validation_interval":40}}
def package65(model,m):
    b=io.BytesIO(); torch.save(model.state_dict(),b); raw=b.getvalue(); state=torch.load(io.BytesIO(raw),map_location="cpu",weights_only=True); ms=sha65(canonical65(m).encode()); ss=sd65(state); rs=sha65(raw); bd=sha65(canonical65([ms,ss,rs]).encode()); return {"manifest":copy.deepcopy(m),"manifest_sha":ms,"state_bytes":raw,"state_digest":ss,"state_bytes_sha":rs,"bundle_digest":bd}
pkg65=package65(model65,manifest65); REG65=MappingProxyType({("nbeats-synthetic-v1",1):pkg65["bundle_digest"]})
def make_model65(): return NBeats65([NBeatsBlock65(BACK65,TrendBasis65(BACK65,HORIZON65,2)),NBeatsBlock65(BACK65,SeasonalityBasis65(BACK65,HORIZON65,4)),NBeatsBlock65(BACK65,GenericBasis65(BACK65,HORIZON65))],HORIZON65)
class PublishedNBeats65:
    def __init__(self,model,mean,std): self._model=model; self._mean=float(mean); self._std=float(std)
    @torch.no_grad()
    def forecast(self,history):
        x=torch.as_tensor(history,dtype=torch.float32)
        if x.ndim!=2 or x.shape[0]<1 or x.shape[1]!=BACK65 or not torch.isfinite(x).all(): raise ValueError("published_history_contract")
        return self._model((x-self._mean)/self._std)*self._std+self._mean
def load65(pkg):
    m=pkg["manifest"]; key=(m.get("artifact_id"),m.get("version"))
    regen=make_series65(m["data"]["length"],m["data"]["seed"]); dm=regen[:m["data"]["train_end"]].mean(); ds=regen[:m["data"]["train_end"]].std(unbiased=False)
    if m!=manifest65 or th65(regen)!=m["data"]["snapshot"] or not torch.isclose(dm,torch.tensor(m["preprocess"]["mean"])) or not torch.isclose(ds,torch.tensor(m["preprocess"]["std"])): raise RuntimeError("manifest_data_contract")
    current_ms=sha65(canonical65(m).encode()); current_rs=sha65(pkg["state_bytes"])
    if current_ms!=pkg["manifest_sha"] or current_rs!=pkg["state_bytes_sha"]: raise RuntimeError("state_bytes_contract")
    state=torch.load(io.BytesIO(pkg["state_bytes"]),map_location="cpu",weights_only=True)
    current_sd=sd65(state)
    if current_sd!=pkg["state_digest"]: raise RuntimeError("state_digest_contract")
    current_bundle=sha65(canonical65([current_ms,current_sd,current_rs]).encode())
    if pkg.get("bundle_digest")!=current_bundle: raise RuntimeError("bundle_contract")
    if REG65.get(key)!=current_bundle: raise RuntimeError("publisher_registry_rejected")
    model=make_model65(); model.load_state_dict(state); model.eval(); return PublishedNBeats65(model,dm,ds)
pub65=load65(pkg65); raw_history65=raw65[test_o65[:3,None]-BACK65+torch.arange(BACK65)]
served65=pub65.forecast(raw_history65); direct65=model65(test_x65[:3])*std65+mean65
assert torch.allclose(served65,direct65,atol=1e-6)
forged65=package65(make_model65(),manifest65)
try: load65(forged65); raise AssertionError("re-signed forecaster accepted")
except RuntimeError as e: assert str(e)=="publisher_registry_rejected"
forged_old_bundle65=copy.deepcopy(forged65); forged_old_bundle65["bundle_digest"]=pkg65["bundle_digest"]
try: load65(forged_old_bundle65); raise AssertionError("forged forecaster with old bundle accepted")
except RuntimeError as e: assert str(e)=="bundle_contract"

### 6.1 服务负例与篡改测试

一个可发布预测器必须把失败行为也定义清楚：空历史、错长度和 NaN 要拒绝；同一输入要得到逐位一致的结果；manifest 或权重字节被修改时必须 fail closed。这里只信任代码外的只读 registry，因此攻击者即使替换权重并重新计算包内摘要，也不能把它伪装成已批准版本。

生产系统还应把告警阈值按 horizon 拆开，并记录输入缺失率、漂移、预测延迟和回填后的真实误差，避免只监控平均 loss。

In [ ]:
assert served65.shape==(3,HORIZON65) and served65.dtype==torch.float32 and torch.isfinite(served65).all()
assert torch.equal(served65,pub65.forecast(raw_history65))
assert isinstance(REG65,MappingProxyType) and len(REG65)==1
try: pub65.forecast(torch.zeros(2,BACK65-1)); raise AssertionError("wrong published window accepted")
except ValueError as e: assert str(e)=="published_history_contract"
try: pub65.forecast(torch.full((1,BACK65),float("nan"))); raise AssertionError("NaN history accepted")
except ValueError as e: assert str(e)=="published_history_contract"
try: pub65.forecast(torch.empty(0,BACK65)); raise AssertionError("empty history batch accepted")
except ValueError as e: assert str(e)=="published_history_contract"
tampered_manifest65=copy.deepcopy(pkg65); tampered_manifest65["manifest"]["training"]["steps"]+=1
try: load65(tampered_manifest65); raise AssertionError("tampered manifest accepted")
except RuntimeError as e: assert str(e)=="manifest_data_contract"
damaged_bytes65=copy.deepcopy(pkg65); damaged_bytes65["state_bytes"]=damaged_bytes65["state_bytes"]+b"x"
try: load65(damaged_bytes65); raise AssertionError("damaged bytes accepted")
except RuntimeError as e: assert str(e)=="state_bytes_contract"
state_copy65={k:v.clone() for k,v in model65.state_dict().items()}; first_key65=next(iter(state_copy65)); state_copy65[first_key65].view(-1)[0]+=1
assert sd65(state_copy65)!=sd65(model65.state_dict())

## 7. 失败模式、复杂度与来源

常见错误：随机切分窗口；全数据标准化；forecast target 越过 split；把 component 当因果分解；漏减 backcast；递归多步却按直接多步评估；只和零基线比较。MLP 计算约随 block 数与 hidden 平方增长，窗口数据复制也可能成为内存瓶颈。

- Oreshkin et al., [N-BEATS](https://arxiv.org/abs/1905.10437), ICLR 2020。
- Olivares et al., [NeuralForecast](https://arxiv.org/abs/2202.12852)，工程化复现背景。
- Hyndman & Athanasopoulos, [Forecasting: Principles and Practice](https://otexts.com/fpp3/)，时间评估背景。